# CambaPredict - Modelo de Machine Learning para Tiempos de Entrega

Este cuaderno entrena un modelo de Machine Learning para predecir el tiempo de entrega de pedidos urbanos en Santa Cruz de la Sierra.

Los datos están en `entregas_camba.csv`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

# Cargar datos
df = pd.read_csv('entregas_camba.csv')
df.head()

In [ ]:
# Preprocesamiento
# Convertir variable categórica 'vehiculo' usando One-Hot Encoding
df = pd.get_dummies(df, columns=['vehiculo'], drop_first=True)

# Separar variables independientes (X) y variable objetivo (y)
X = df.drop('tiempo_real_min', axis=1)
y = df['tiempo_real_min']

# Partición de entrenamiento y prueba (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# 1. Regresión Lineal
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)
lr_pred = lr_model.predict(X_test)

# 2. Árbol de Decisión
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

# 3. Random Forest
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

In [ ]:
# Función para evaluar modelos
def evaluar(y_true, y_pred, modelo):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    print(f"--- {modelo} ---")
    print(f"MAE: {mae:.2f} min")
    print(f"RMSE: {rmse:.2f} min")
    print(f"R²: {r2:.2f}\n")

evaluar(y_test, lr_pred, "Regresión Lineal")
evaluar(y_test, dt_pred, "Árbol de Decisión")
evaluar(y_test, rf_pred, "Random Forest")

In [ ]:
# Detección de Overfitting en Random Forest
rf_train_pred = rf_model.predict(X_train)
rmse_train = np.sqrt(mean_squared_error(y_train, rf_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, rf_pred))

print(f"RMSE Entrenamiento: {rmse_train:.2f} min")
print(f"RMSE Validación: {rmse_test:.2f} min")

In [ ]:
# Importancia de las variables (Random Forest)
importancias = pd.Series(rf_model.feature_importances_, index=X.columns)
importancias.sort_values().plot(kind='barh', title='Importancia de variables')
plt.show()

In [ ]:
# Guardar el mejor modelo (Random Forest)
with open('modelo_camba.pkl', 'wb') as f:
    pickle.dump(rf_model, f)
print("Modelo Random Forest guardado como 'modelo_camba.pkl'")

In [ ]:
# Guardar los nombres de las columnas usadas en entrenamiento
with open('columnas_modelo.pkl', 'wb') as f:
    pickle.dump(list(X.columns), f)
print("Columnas del modelo guardadas.")